# Bayesian Spatial Modelling of Seismic Activity — Japan
### Complete analysis notebook

**Team:** Paweł Majerczyk, Jakub Gicala — Data Analytics (AGH, AIR ISZ)

This single notebook covers the full pipeline:
1. [Data acquisition](#1-data-acquisition)
2. [Preprocessing & EDA](#2-preprocessing--eda)
3. [Priors — rationale, DAG, prior predictive checks](#3-priors)
4. [Model 1 — no-pooling posterior](#4-model-1--no-pooling-posterior)
5. [Model 2 — partial-pooling posterior](#5-model-2--partial-pooling-posterior)
6. [Model comparison](#6-model-comparison)

In [ ]:
import sys, os, warnings
sys.path.append("..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import arviz as az
from cmdstanpy import CmdStanModel

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110
az.style.use("arviz-darkgrid")

from utils.display import display_df, display_image
from utils.data_prep import build_stan_data, LAT_BINS, LON_BINS, PRIOR_MU, PRIOR_SIGMA_M1

RNG = np.random.default_rng(2024)

---
## 1. Data Acquisition

**What we do:** load the raw earthquake catalogue downloaded from the USGS
Earthquake Catalog API for Japan and describe the data source and its columns.

**Why:** the whole project models seismic activity, so we first need a clean,
documented record of *where* and *when* earthquakes happened. This notebook
establishes provenance (where the data comes from), shows the variables we have,
and produces a first geographic picture of the raw events.

### The phenomenon and use case

Japan sits at the junction of four tectonic plates (Pacific, Philippine Sea,
Eurasian, North American) and is one of the most seismically active regions on
Earth. We model **annual counts of M ≥ 4.0 earthquakes per 2°×2° grid cell**
over the bounding box 24–50°N, 122–154°E (2000–2023). The output is an
uncertainty-aware **posterior intensity map** — useful for seismic-hazard
assessment, insurance pricing, and infrastructure planning.

### Data source — USGS Earthquake Catalog

Data: [https://earthquake.usgs.gov/fdsnws/event/1/](https://earthquake.usgs.gov/fdsnws/event/1/)
(CSV, no API key required). Downloaded year-by-year (20 000-record API limit).

- **Coverage:** 2000-01-01 to 2023-12-31
- **Filter:** M ≥ 4.0, lat 24–50°N, lon 122–154°E
- **Total events:** ~33 000

#### Columns of interest

| Column | Description |
|---|---|
| `time` | ISO-8601 UTC timestamp of the event |
| `latitude` / `longitude` | Epicentre coordinates (decimal degrees) |
| `depth` | Hypocentre depth (km) |
| `mag` | Moment magnitude |
| `id` | USGS event identifier |

All other columns (`magType`, `nst`, `gap`, …) are quality metadata and are not
used in modelling.

In [ ]:
RAW_PATH = "../data/raw/earthquakes_japan.csv"
df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

### Basic statistics

In [ ]:
df["year"] = df["time"].str[:4].astype(int)

print("Time span      :", df["year"].min(), "->", df["year"].max())
print("Magnitude range:", df["mag"].min(), "->", df["mag"].max())
print("Latitude range :", round(df["latitude"].min(),2), "->", round(df["latitude"].max(),2))
print("Longitude range:", round(df["longitude"].min(),2), "->", round(df["longitude"].max(),2))
print("\nMissing values in key columns:")
print(df[["latitude","longitude","mag","time"]].isnull().sum())

display_df(df[["depth","mag","latitude","longitude"]].describe().round(2),
           caption="Key variable summaries")

In [ ]:
# Events per year — note the huge spike in 2011 (Tohoku M9.0 + aftershocks).
yearly = df.groupby("year").size().reset_index(name="count")
fig, ax = plt.subplots()
ax.bar(yearly["year"], yearly["count"], color="steelblue", edgecolor="k", linewidth=0.4)
ax.axhline(yearly["count"].median(), color="orange", ls="--", label=f'Median {yearly["count"].median():.0f}')
ax.set_xlabel("Year"); ax.set_ylabel("Events (M≥4.0)")
ax.set_title("Annual event count — Japan 2000-2023")
ax.legend(); plt.tight_layout()
plt.savefig("../report/figures/02_annual_counts.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"2011 count: {yearly.loc[yearly.year==2011,'count'].values[0]}  "
      f"(median other years: {yearly.loc[yearly.year!=2011,'count'].median():.0f})")

### Map of raw events

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
sc = ax.scatter(df["longitude"], df["latitude"],
                c=df["mag"], cmap="YlOrRd", s=1.5, alpha=0.4, vmin=4, vmax=8)
plt.colorbar(sc, ax=ax, label="Magnitude")
ax.set_xlim(122, 154); ax.set_ylim(24, 50)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Raw earthquake epicentres — Japan 2000-2023 (M≥4.0)")
os.makedirs("../report/figures", exist_ok=True)
plt.savefig("../report/figures/01_raw_map.png", dpi=130, bbox_inches="tight")
plt.show()
display_image("../report/figures/01_raw_map.png", width=520)

**Summary.** The catalogue is dense along the eastern coast (Pacific subduction
zones) and sparse in the west. Year 2011 is an extreme outlier driven by the
M9.0 Tohoku earthquake and its aftershock sequence. This stationarity issue will
resurface in the model-comparison section.

---
## 2. Preprocessing & EDA

**What we do:** aggregate point epicentres to a 2°×2° grid and produce one
Poisson count per cell-year — the model input.

In [ ]:
# ---- Grid assignment -------------------------------------------------------
df["lat_idx"] = pd.cut(df["latitude"],  bins=LAT_BINS, labels=False, right=False)
df["lon_idx"] = pd.cut(df["longitude"], bins=LON_BINS, labels=False, right=False)
df = df.dropna(subset=["lat_idx","lon_idx"])
df["lat_idx"] = df["lat_idx"].astype(int)
df["lon_idx"] = df["lon_idx"].astype(int)
df["cell_id"] = df["lat_idx"].astype(str) + "_" + df["lon_idx"].astype(str)

n_cells_total = (len(LAT_BINS)-1) * (len(LON_BINS)-1)
print(f"Grid: {len(LAT_BINS)-1} rows × {len(LON_BINS)-1} cols = {n_cells_total} cells")
print(f"Active cells (≥1 event): {df['cell_id'].nunique()}")

In [ ]:
# ---- Annual counts per cell ------------------------------------------------
annual = (df.groupby(["cell_id","year","lat_idx","lon_idx"])
            .size().reset_index(name="count"))
annual["lat_center"] = LAT_BINS[annual["lat_idx"]] + 1.0
annual["lon_center"] = LON_BINS[annual["lon_idx"]] + 1.0

# Fill missing cell-years with 0 (cells exist even in quiet years)
all_cells = annual[["cell_id","lat_idx","lon_idx","lat_center","lon_center"]].drop_duplicates()
all_years  = pd.DataFrame({"year": range(df["year"].min(), df["year"].max()+1)})
full = all_cells.merge(all_years, how="cross")
annual = full.merge(annual, on=["cell_id","year","lat_idx","lon_idx","lat_center","lon_center"], how="left")
annual["count"] = annual["count"].fillna(0).astype(int)

os.makedirs("../data/processed", exist_ok=True)
annual.to_csv("../data/processed/grid_annual_counts.csv", index=False)
print(f"Saved grid_annual_counts.csv: {annual.shape} (cell-year rows)")
print(f"Active cells: {(annual.groupby('cell_id')['count'].sum() > 0).sum()}")

In [ ]:
# ---- Cell metadata ---------------------------------------------------------
meta = (annual.groupby(["cell_id","lat_idx","lon_idx","lat_center","lon_center"])
              .agg(total_events=("count","sum"),
                   n_years=("year","count"),
                   mean_count=("count","mean"))
              .reset_index())
meta.to_csv("../data/processed/grid_metadata.csv", index=False)
display_df(meta.sort_values("total_events", ascending=False).head(10),
           caption="Top-10 most active cells")

### EDA 1 — distribution of annual counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(annual["count"], bins=60, color="steelblue", edgecolor="k")
axes[0].set_title("Annual count per cell (linear)")
axes[0].set_xlabel("events / cell / year"); axes[0].set_ylabel("frequency")

axes[1].hist(np.log1p(annual["count"]), bins=40, color="darkorange", edgecolor="k")
axes[1].set_title("Annual count per cell (log1p)")
axes[1].set_xlabel("log(1 + count)"); axes[1].set_ylabel("frequency")
plt.tight_layout(); plt.show()

### EDA 2 — raw epicentres overlaid on the grid

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)

# Draw grid lines
for lat in LAT_BINS:
    ax.axhline(lat, color="grey", lw=0.4, alpha=0.6)
for lon in LON_BINS:
    ax.axvline(lon, color="grey", lw=0.4, alpha=0.6)

# Colour cells by mean annual count
for _, row in meta.iterrows():
    lat0 = LAT_BINS[int(row["lat_idx"])]
    lon0 = LON_BINS[int(row["lon_idx"])]
    rect = plt.Rectangle((lon0, lat0), 2, 2,
                          facecolor=plt.cm.YlOrRd(min(row["mean_count"] / 30, 1.0)),
                          edgecolor="grey", lw=0.4, alpha=0.5)
    ax.add_patch(rect)

# Raw epicentres on top
ax.scatter(df["longitude"], df["latitude"],
           s=0.8, alpha=0.25, color="navy", linewidths=0)

ax.set_xlim(122, 154); ax.set_ylim(24, 50)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Raw epicentres + 2°×2° grid coloured by mean annual count")

import matplotlib.cm as cm
import matplotlib.colors as mcolors
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=mcolors.Normalize(vmin=0, vmax=30))
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Mean events / year (cell colour)")
plt.show()

### EDA 3 — mean activity per cell (heatmap)

In [ ]:
grid_mean = np.zeros((len(LAT_BINS)-1, len(LON_BINS)-1))
for _, row in meta.iterrows():
    grid_mean[int(row["lat_idx"]), int(row["lon_idx"])] = row["mean_count"]

fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
im = ax.imshow(grid_mean, origin="lower", aspect="auto",
               extent=[LON_BINS[0], LON_BINS[-1], LAT_BINS[0], LAT_BINS[-1]],
               cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="Mean events / year")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Mean annual earthquake count per 2°×2° cell (2000-2023)")
plt.show()

---
## 3. Priors

**What we do:** choose and justify the priors for both models, derive the prior
parameters from *external* knowledge only (no data-leakage), visualise the DAG,
and run two prior predictive checks.

### 3.1 Archival data as the basis for priors — split-in-time argument

A prior must not be derived from the same data used in the likelihood — that
would be **double-dipping** (the same observations would influence the posterior
twice: once through the prior and once through the likelihood), leading to
over-confident posteriors and invalid uncertainty quantification.

#### Why the historical rate ~1200 M≥4/yr is a valid prior source

Our analysis covers **2000–2023** (USGS). The rate ~1200 M≥4 events per year
for Japan is drawn from **long-running independent catalogues** that substantially
pre-date our analysis window:

| Source | Period | Rate |
|---|---|---|
| Japan Meteorological Agency (JMA) catalog | 1923–present | ~1000–1400/yr |
| International Seismological Centre (ISC) | 1960–present | consistent |
| USGS/NEIC long-term average (pre-2000) | ~1970–1999 | ~1150/yr |
| Published literature (e.g. Utsu 1971, Kasahara 1981) | decades of data | 1100–1300/yr |

This is a **split-in-time** design: the prior is informed by the *historical*
record (pre-2000), while the likelihood is formed on the *test* period (2000–2023).
There is no overlap. This is the standard approach used in operational seismic
hazard assessment (cf. Gutenberg & Richter 1944; Räty et al. 2023, NHESS;
Tu et al. 2025, Annals of GIS).

#### Prior centre derivation

```
Published Japan rate:  ~1200 M≥4 events / year     (external, split-in-time)
Grid geometry:          13 lat bins × 16 lon bins = 208 cells   (pure geometry)
Rate per cell:          1200 / 208 ≈ 5.77 events / year
Prior centre:           mu_0 = log(5.77) ≈ 1.75 ≈ 1.8
```

#### Prior scale derivation (Gutenberg-Richter)

The Gutenberg-Richter law (log₁₀N(≥M) = a − b·M, b ≈ 1 for Japan) constrains
the plausible range of log-intensities. A *very quiet* cell might see ~1 event/yr
(log ≈ 0); an extreme Tohoku-scale aftershock cluster can reach ~1000–3000/yr
(log ≈ 7–8). Spanning this range in ±3σ:

```
sigma_0 = (log(3000) - log(1)) / 6  ≈  8.0 / 6  ≈  1.33    [conservative]
sigma_0 = (8 - mu_0) / 3            ≈  (8 - 1.8) / 3 ≈ 2.07  [used here]
```

We use σ₀ = 2.07, which places the 99.9th-percentile cell at ~3700 events/yr —
comfortably above the worst observed year (2011 Tohoku, ~1400 events in the
hottest cell) without being implausibly wide.

In [ ]:
# External-knowledge derivation — no sample statistics used
JAPAN_RATE_M4 = 1200          # published long-term rate (JMA/ISC, pre-2000)
n_rows  = len(LAT_BINS) - 1   # 13  — pure geometry
n_cols  = len(LON_BINS) - 1   # 16  — pure geometry
n_cells = n_rows * n_cols     # 208

rate_per_cell = JAPAN_RATE_M4 / n_cells       # ~5.77 events/yr
mu0           = np.log(rate_per_cell)          # ~1.75 ≈ 1.8 (used as PRIOR_MU)
sigma0        = (8 - mu0) / 3                  # ~2.08 ≈ 2.07 (used as PRIOR_SIGMA_M1)

print(f"Grid geometry  : {n_rows}×{n_cols} = {n_cells} cells")
print(f"Rate per cell  : {rate_per_cell:.2f} events/yr")
print(f"mu_0 = log({rate_per_cell:.2f}) = {mu0:.3f}  (stored as PRIOR_MU={PRIOR_MU})")
print(f"sigma_0 = (8 - {mu0:.2f}) / 3  = {sigma0:.3f}  (stored as PRIOR_SIGMA_M1={PRIOR_SIGMA_M1})")
print()
print("NOTE: no sample mean/median/std from the count data is used above.")
print("      This derivation uses only: (1) published Japan rate, (2) grid geometry.")

### 3.2 Directed Acyclic Graph (DAG)

A DAG shows the assumed causal/generative structure — which parameters generate
which observations, and which hyperparameters generate which parameters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---- DAG: Model 1 (no pooling) ---------------------------------------------
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis("off")
ax.set_title("Model 1 — No Pooling", fontsize=13, fontweight="bold")

nodes_m1 = {
    "mu_0\n1.8":       (5, 7.0),
    "sigma_0\n2.07":   (7, 7.0),
    "alpha_c":         (5, 5.0),
    "lambda_c":        (5, 3.0),
    "count_{c,y}":     (5, 1.0),
}
edges_m1 = [
    ("mu_0\n1.8",    "alpha_c"),
    ("sigma_0\n2.07","alpha_c"),
    ("alpha_c",      "lambda_c"),
    ("lambda_c",     "count_{c,y}"),
]

# plate annotation
plate_m1 = plt.Rectangle((3.2, 0.3), 3.6, 5.2,
                          fill=False, edgecolor="grey", linewidth=1.5, linestyle="--")
ax.add_patch(plate_m1)
ax.text(6.7, 0.5, "c = 1…154\ny = 2000…2023", fontsize=7, color="grey", ha="right")

for label, (x, y) in nodes_m1.items():
    color = "#d0e8ff" if label in ("alpha_c", "lambda_c") else \
            "#ffe0b2" if "count" in label else "#e8f5e9"
    ax.add_patch(plt.Circle((x, y), 0.55, color=color, ec="black", lw=1.2, zorder=3))
    ax.text(x, y, label, ha="center", va="center", fontsize=8, zorder=4)

for (src, dst) in edges_m1:
    x0, y0 = nodes_m1[src]; x1, y1 = nodes_m1[dst]
    ax.annotate("", xy=(x1, y1+0.55), xytext=(x0, y0-0.55),
                arrowprops=dict(arrowstyle="->", color="black", lw=1.2))

ax.text(1.0, 6.5, "Fixed\nhyperparameters", fontsize=8, color="#555")
ax.text(1.0, 5.0, "Cell log-\nintensity", fontsize=8, color="#555")
ax.text(1.0, 3.0, "Poisson\nrate", fontsize=8, color="#555")
ax.text(1.0, 1.0, "Observed\ncount", fontsize=8, color="#555")

# ---- DAG: Model 2 (partial pooling) ----------------------------------------
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis("off")
ax.set_title("Model 2 — Partial Pooling (Hierarchical)", fontsize=13, fontweight="bold")

nodes_m2 = {
    "mu_0\n1.8":          (3.5, 7.2),
    "sigma_hp\n1.0":      (6.5, 7.2),
    "mu_global":          (3.5, 5.5),
    "sigma_global":       (6.5, 5.5),
    "alpha_c":            (5.0, 3.8),
    "lambda_c":           (5.0, 2.3),
    "count_{c,y}":        (5.0, 0.7),
}
edges_m2 = [
    ("mu_0\n1.8",     "mu_global"),
    ("sigma_hp\n1.0", "sigma_global"),
    ("mu_global",     "alpha_c"),
    ("sigma_global",  "alpha_c"),
    ("alpha_c",       "lambda_c"),
    ("lambda_c",      "count_{c,y}"),
]

plate_m2 = plt.Rectangle((3.2, 0.1), 3.6, 4.3,
                          fill=False, edgecolor="grey", linewidth=1.5, linestyle="--")
ax.add_patch(plate_m2)
ax.text(6.7, 0.3, "c = 1…154\ny = 2000…2023", fontsize=7, color="grey", ha="right")

for label, (x, y) in nodes_m2.items():
    color = "#d0e8ff" if label in ("alpha_c","lambda_c","mu_global","sigma_global") else \
            "#ffe0b2" if "count" in label else "#e8f5e9"
    ax.add_patch(plt.Circle((x, y), 0.55, color=color, ec="black", lw=1.2, zorder=3))
    ax.text(x, y, label, ha="center", va="center", fontsize=8, zorder=4)

for (src, dst) in edges_m2:
    x0, y0 = nodes_m2[src]; x1, y1 = nodes_m2[dst]
    ax.annotate("", xy=(x1, y1+0.55), xytext=(x0, y0-0.55),
                arrowprops=dict(arrowstyle="->", color="black", lw=1.2))

ax.text(0.3, 7.0, "Fixed hyper-\nparameters", fontsize=8, color="#555")
ax.text(0.3, 5.2, "Estimated\nhyperparameters", fontsize=8, color="#555")
ax.text(0.3, 3.6, "Cell log-\nintensity", fontsize=8, color="#555")
ax.text(0.3, 0.7, "Observed\ncount", fontsize=8, color="#555")

plt.suptitle("Generative DAGs — both models share the same likelihood;\n"
             "the structural difference is whether σ is fixed or estimated.",
             fontsize=10, y=0.02)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig("../report/figures/03_dag.png", dpi=130, bbox_inches="tight")
plt.show()

### 3.3 Prior predictive check 1 — parameter space

We draw α from the prior and inspect the implied λ = exp(α). This checks that
the prior assigns negligible mass to physically impossible values (λ < 0 is
impossible for Poisson; λ > several thousand is implausible for a 2°×2° cell).

In [ ]:
NSIM = 20_000
# Model 1 fixed prior
alpha_m1 = RNG.normal(PRIOR_MU, PRIOR_SIGMA_M1, NSIM)
lam_m1   = np.exp(alpha_m1)

# Model 2 hierarchical prior (draw hyperparams, then alpha)
mu_g    = RNG.normal(PRIOR_MU, 1, NSIM)
sig_g   = np.abs(RNG.normal(0, 1, NSIM))
alpha_m2 = RNG.normal(mu_g, sig_g)
lam_m2   = np.exp(alpha_m2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lam, title in zip(axes, [lam_m1, lam_m2], ["Model 1 prior", "Model 2 prior"]):
    ax.hist(np.log1p(lam), bins=60, color="grey", edgecolor="k", alpha=0.7)
    ax.set_xlabel("log(1 + λ)"); ax.set_ylabel("frequency")
    ax.set_title(title + " — λ distribution")
    ax.axvline(np.log1p(1),    color="blue",  ls="--", label="λ=1  (1/yr)")
    ax.axvline(np.log1p(1200), color="red",   ls="--", label="λ=1200 (Japan total/cell)")
    ax.axvline(np.log1p(3000), color="purple",ls=":",  label="λ=3000 (Tohoku scale)")
    ax.legend(fontsize=7)

print(f"M1 prior — median λ: {np.median(lam_m1):.1f},  "
      f"99.9th pctile: {np.percentile(lam_m1,99.9):.0f},  "
      f"frac > 3000: {(lam_m1>3000).mean():.4f}")
print(f"M2 prior — median λ: {np.median(lam_m2):.1f},  "
      f"99.9th pctile: {np.percentile(lam_m2,99.9):.0f},  "
      f"frac > 3000: {(lam_m2>3000).mean():.4f}")
plt.tight_layout(); plt.show()

### 3.4 Prior predictive check 2 — measurement space

We simulate complete datasets (one count per cell-year) from the prior and
compare the simulated distribution of counts against the observed distribution.
This is a *plausibility* check only — it does not influence the prior parameters
(which were fixed by external knowledge above).

In [ ]:
annual = pd.read_csv("../data/processed/grid_annual_counts.csv")  # post-hoc overlay only
stan_data, cells, cell_to_int = build_stan_data(annual)
N = stan_data["N"]

# Simulate counts from the prior predictive
counts_prior_sim = []
for _ in range(200):
    alpha_draw = RNG.normal(PRIOR_MU, PRIOR_SIGMA_M1, N)
    lam_draw   = np.exp(alpha_draw)
    counts_prior_sim.append(RNG.poisson(lam_draw))

counts_sim_flat = np.concatenate(counts_prior_sim)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log1p(counts_sim_flat), bins=50, density=True,
        color="grey", alpha=0.5, label="Prior predictive (200 datasets)")
ax.hist(np.log1p(stan_data["count"]), bins=50, density=True,
        color="steelblue", alpha=0.6, label="Observed counts")
ax.set_xlabel("log(1 + count)"); ax.set_ylabel("density")
ax.set_title("Prior predictive check 2 — simulated vs observed counts")
ax.legend(); plt.tight_layout()
plt.savefig("../report/figures/04_prior_predictive_check.png", dpi=130, bbox_inches="tight")
plt.show()

### 3.5 Prior sensitivity analysis

We re-fit Model 1 under five different prior specifications to check that the
posterior is not driven by the prior choice (cf. Räty et al. 2023, WAMBS checklist).

In [ ]:
scenarios = [
    ("Default (1.8, 2.07)", 1.8, 2.07),
    ("Centre +1 (2.8, 2.07)", 2.8, 2.07),
    ("Centre -1 (0.8, 2.07)", 0.8, 2.07),
    ("SD halved (1.8, 1.04)", 1.8, 1.04),
    ("SD doubled (1.8, 4.14)", 1.8, 4.14),
]

busiest = pd.read_csv("../data/processed/grid_metadata.csv").sort_values("total_events").iloc[-1]["cell_id"]
poorest = pd.read_csv("../data/processed/grid_metadata.csv").sort_values("total_events").iloc[0]["cell_id"]
mid_cell = pd.read_csv("../data/processed/grid_metadata.csv").sort_values("total_events").iloc[len(pd.read_csv("../data/processed/grid_metadata.csv"))//2]["cell_id"]

model1_stan = CmdStanModel(stan_file="../models/model1_nopool.stan")
results = {}
for name, mu, sigma in scenarios:
    sd_local = stan_data.copy()
    sd_local["mu_prior_mean"] = mu
    sd_local["sigma_fixed"]   = sigma
    fit = model1_stan.sample(data=sd_local, chains=2, parallel_chains=2,
                             iter_warmup=500, iter_sampling=500, seed=2024,
                             show_progress=False)
    idata = az.from_cmdstanpy(
        posterior=fit, observed_data={"count": sd_local["count"]},
        coords={"cell": cells}, dims={"alpha": ["cell"]})
    results[name] = idata

print("Sensitivity analysis complete. Comparing posterior means for 3 representative cells:")
print(f"{'Scenario':<30} {'Busy cell':>12} {'Mid cell':>12} {'Poor cell':>12}")
for name, idata in results.items():
    alpha_post = idata.posterior["alpha"].mean(("chain","draw")).values
    cell_list  = list(cells)
    b = np.exp(alpha_post[cell_list.index(busiest)])
    m = np.exp(alpha_post[cell_list.index(mid_cell)])
    p = np.exp(alpha_post[cell_list.index(poorest)])
    print(f"{name:<30} {b:>12.1f} {m:>12.1f} {p:>12.1f}")

---
## 4. Model 1 — No-Pooling Posterior

**What we do:** fit the no-pooling Poisson model with CmdStanPy, check
convergence, and analyse the posterior.

$$\text{count}_{c,y}\sim\text{Poisson}(\lambda_c),\quad \log\lambda_c=\alpha_c,
\quad \alpha_c \sim \mathcal{N}(\mu_0=1.8,\ \sigma_0=2.07)\ \text{(fixed)}.$$

The fixed $\sigma_0$ is the defining feature: there is no hyperparameter linking
cells, so this is genuinely *no pooling*.

In [ ]:
annual = pd.read_csv("../data/processed/grid_annual_counts.csv")
meta   = pd.read_csv("../data/processed/grid_metadata.csv")
stan_data, cells, cell_to_int = build_stan_data(annual)
print(f"N (cell-year obs) = {stan_data['N']},  C (cells) = {stan_data['C']}")
print(f"Prior: alpha ~ Normal({stan_data['mu_prior_mean']}, {stan_data['sigma_fixed']}) [fixed]")

In [ ]:
model1 = CmdStanModel(stan_file="../models/model1_nopool.stan")
fit1   = model1.sample(data=stan_data, chains=4, parallel_chains=4,
                       iter_warmup=1000, iter_sampling=1000, seed=2024,
                       show_progress=False)
print(fit1.diagnose())

In [ ]:
idata1 = az.from_cmdstanpy(
    posterior=fit1,
    posterior_predictive="count_pred",
    log_likelihood="log_lik",
    observed_data={"count": stan_data["count"]},
    coords={"cell": cells, "obs": np.arange(stan_data["N"])},
    dims={"alpha": ["cell"], "lambda": ["cell"],
          "count_pred": ["obs"], "log_lik": ["obs"]},
)

### Convergence diagnostics

In [ ]:
summ  = az.summary(idata1, var_names=["alpha"])
n_div = int(idata1.sample_stats["diverging"].values.sum())
print("Divergent transitions:", n_div)
print(f"max R-hat    = {summ['r_hat'].max():.4f}   (target < 1.01)")
print(f"min ESS bulk = {summ['ess_bulk'].min():.0f}")
print(f"min ESS tail = {summ['ess_tail'].min():.0f}")
display_df(summ.sort_values("r_hat", ascending=False).head(8),
           caption="Worst-converging cells (by R-hat)")

In [ ]:
busiest = meta.sort_values("total_events", ascending=False).iloc[0]["cell_id"]
poorest = meta.sort_values("total_events").iloc[0]["cell_id"]
mid     = meta.sort_values("total_events").iloc[len(meta)//2]["cell_id"]
rep_cells = [busiest, mid, poorest]
print("Representative cells (busiest, mid, poorest):", rep_cells)
print(meta.set_index("cell_id").loc[rep_cells, ["total_events","n_years","mean_count"]])

### Trace plots

In [ ]:
az.plot_trace(idata1, var_names=["alpha"],
              coords={"cell": rep_cells}, compact=False)
plt.tight_layout(); plt.show()

### Marginal posteriors

In [ ]:
az.plot_posterior(idata1, var_names=["alpha"],
                  coords={"cell": rep_cells})
plt.tight_layout(); plt.show()

In [ ]:
# Scatter: data richness vs posterior concentration
cell_totals = meta.set_index("cell_id")["total_events"]
alpha_sd    = summ["sd"]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(cell_totals.values, alpha_sd.values, s=15, alpha=0.6, color="steelblue")
ax.set_xlabel("Total events in cell (all years)")
ax.set_ylabel("Posterior SD of alpha")
ax.set_title("Model 1 — data richness vs posterior uncertainty")
ax.set_xscale("log"); plt.tight_layout(); plt.show()

### Posterior predictive check

In [ ]:
obs    = stan_data["count"]
pp     = idata1.posterior_predictive["count_pred"].values.reshape(-1, stan_data["N"])
pp_mean = pp.mean(axis=0)
pp_low  = np.percentile(pp, 2.5, axis=0)
pp_high = np.percentile(pp, 97.5, axis=0)

# Top-15 cells by total events
top_cells_idx = np.argsort(
    [cell_totals.get(cells[cell_to_int[c]-1], 0)
     if c in cell_to_int else 0
     for c in cells]
)[-15:][::-1]

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(top_cells_idx))
ax.bar(x - 0.2, [obs[top_cells_idx[i]] for i in range(len(top_cells_idx))],
       0.4, label="Observed (mean)", color="steelblue")
ax.bar(x + 0.2, [pp_mean[top_cells_idx[i]] for i in range(len(top_cells_idx))],
       0.4, label="Posterior predictive mean", color="orange", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels([cells[i] for i in top_cells_idx], rotation=45, ha="right")
ax.set_ylabel("Events"); ax.set_title("Model 1 — PPC: observed vs predicted (top 15 cells)")
ax.legend(); plt.tight_layout(); plt.show()

### Data consistency — 2011 Tohoku outlier

In [ ]:
resid = np.abs(obs - pp_mean)
worst_idx = np.argsort(resid)[-10:][::-1]
print("Worst-fit observations (|obs - pp_mean|):")
obs_df = annual.copy()
for idx in worst_idx:
    print(f"  obs #{idx:4d}: cell={annual.iloc[idx]['cell_id']}, "
          f"year={annual.iloc[idx]['year']}, "
          f"count={obs[idx]}, pp_mean={pp_mean[idx]:.1f}")

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
import pathlib
pathlib.Path("../data/processed/idata_model1.nc").unlink(missing_ok=True)
idata1.to_netcdf("../data/processed/idata_model1.nc", overwrite_existing=True, engine="netcdf4")
print("Saved idata_model1.nc")

**Summary — Model 1.** Log-concave Poisson likelihood → NUTS samples cleanly:
no divergences, all $\hat R < 1.01$, ESS in the thousands. Data-poor cells have
wide posteriors (prior-dominated); data-rich cells are tightly estimated. The
2011 Tohoku cell-years are systematic outliers — a stationary model cannot
capture a one-off regime change.

---
## 5. Model 2 — Partial-Pooling Posterior

**What we do:** fit the hierarchical Poisson model, check convergence, quantify
the shrinkage effect, draw the prior vs posterior map, and produce the final
intensity map.

$$\text{count}_{c,y}\sim\text{Poisson}(\lambda_c),\quad \log\lambda_c=\alpha_c,$$
$$\alpha_c\sim\mathcal{N}(\mu_{\text{global}},\sigma_{\text{global}}),\quad
\mu_{\text{global}}\sim\mathcal{N}(1.8,1),\quad
\sigma_{\text{global}}\sim\text{HalfNormal}(1).$$

The **only structural change** from Model 1 is that $\sigma_{\text{global}}$ is a
free parameter. Because this dataset is *informative* we use the **centered**
parameterization — it mixes better than non-centering in the strong-data regime.

In [ ]:
annual    = pd.read_csv("../data/processed/grid_annual_counts.csv")
meta      = pd.read_csv("../data/processed/grid_metadata.csv")
stan_data, cells, cell_to_int = build_stan_data(annual)
print(f"N = {stan_data['N']}, C = {stan_data['C']}")

In [ ]:
model2 = CmdStanModel(stan_file="../models/model2_partial.stan")
fit2   = model2.sample(data=stan_data, chains=4, parallel_chains=4,
                       iter_warmup=1000, iter_sampling=1000, seed=2024,
                       adapt_delta=0.95, inits={"mu_global": 2.0, "sigma_global": 1.0},
                       show_progress=False)
print(fit2.diagnose())

In [ ]:
idata2 = az.from_cmdstanpy(
    posterior=fit2,
    posterior_predictive="count_pred",
    log_likelihood="log_lik",
    observed_data={"count": stan_data["count"]},
    coords={"cell": cells, "obs": np.arange(stan_data["N"])},
    dims={"alpha": ["cell"], "lambda": ["cell"],
          "count_pred": ["obs"], "log_lik": ["obs"]},
)

### Convergence diagnostics

In [ ]:
summ2  = az.summary(idata2, var_names=["alpha","mu_global","sigma_global"])
n_div2 = int(idata2.sample_stats["diverging"].values.sum())
print("Divergent transitions:", n_div2)
print(f"max R-hat (alpha)  = {az.summary(idata2,var_names=['alpha'])['r_hat'].max():.4f}")
print(f"max R-hat (hypers) = {az.summary(idata2,var_names=['mu_global','sigma_global'])['r_hat'].max():.4f}")
print(f"min ESS bulk       = {summ2['ess_bulk'].min():.0f}")
display_df(az.summary(idata2, var_names=["mu_global","sigma_global"]),
           caption="Hyperparameter summary")

In [ ]:
busiest2 = meta.sort_values("total_events", ascending=False).iloc[0]["cell_id"]
poorest2 = meta.sort_values("total_events").iloc[0]["cell_id"]
mid2     = meta.sort_values("total_events").iloc[len(meta)//2]["cell_id"]
rep_cells2 = [busiest2, mid2, poorest2]

az.plot_trace(idata2, var_names=["mu_global","sigma_global"])
plt.tight_layout(); plt.show()

### Marginal posteriors — hyperparameters and representative cells

In [ ]:
az.plot_posterior(idata2, var_names=["mu_global","sigma_global"])
plt.tight_layout(); plt.show()

In [ ]:
az.plot_posterior(idata2, var_names=["alpha"],
                  coords={"cell": rep_cells2})
plt.tight_layout(); plt.show()

### Posterior predictive check

In [ ]:
obs2   = stan_data["count"]
pp2    = idata2.posterior_predictive["count_pred"].values.reshape(-1, stan_data["N"])
pp2_mean = pp2.mean(axis=0)

in_ci = ((obs2 >= np.percentile(pp2, 2.5, axis=0)) &
         (obs2 <= np.percentile(pp2, 97.5, axis=0)))
print(f"Coverage (95% CI): {in_ci.mean():.1%}  (expect ~95%)")

resid2   = np.abs(obs2 - pp2_mean)
worst2   = np.argsort(resid2)[-8:][::-1]
print("\nWorst-fit observations:")
for idx in worst2:
    print(f"  cell={annual.iloc[idx]['cell_id']}, year={annual.iloc[idx]['year']}, "
          f"count={obs2[idx]}, pp_mean={pp2_mean[idx]:.1f}")

In [ ]:
# Top-15 cells by total events — observed vs predicted (mirrors Model 1 plot)
cell_totals2 = meta.set_index("cell_id")["total_events"]
top_cells_idx2 = np.argsort(
    [cell_totals2.get(cells[cell_to_int[c]-1], 0)
     if c in cell_to_int else 0
     for c in cells]
)[-15:][::-1]

fig, ax = plt.subplots(figsize=(11, 4))
x2 = np.arange(len(top_cells_idx2))
ax.bar(x2 - 0.2, [obs2[top_cells_idx2[i]] for i in range(len(top_cells_idx2))],
       0.4, label="Observed (mean)", color="steelblue")
ax.bar(x2 + 0.2, [pp2_mean[top_cells_idx2[i]] for i in range(len(top_cells_idx2))],
       0.4, label="Posterior predictive mean", color="orange", alpha=0.8)
ax.set_xticks(x2); ax.set_xticklabels([cells[i] for i in top_cells_idx2], rotation=45, ha="right")
ax.set_ylabel("Events"); ax.set_title("Model 2 — PPC: observed vs predicted (top 15 cells)")
ax.legend(); plt.tight_layout(); plt.show()

### 5.4 Prior vs posterior intensity map — both models

The **grey map** (left) shows the shared prior predictive mean intensity (before
seeing any data — identical for both models since they share the same prior
centre). The **middle** and **right** maps show the posterior mean intensity for
Model 1 and Model 2 respectively. Comparing all three reveals what the data
taught each model: where the posterior deviates strongly from the prior, the
data were informative. Model 2 produces a smoother map because data-poor cells
are pulled toward the global mean (shrinkage).

In [ ]:
import geopandas as gpd
from shapely.geometry import box

# ---- Prior predictive median per cell (shared prior, no data) ---------------
# Prior: alpha_c ~ N(PRIOR_MU=1.8, PRIOR_SIGMA_M1=2.07).
# All cells share the SAME prior — there is no geographic information in it.
# We show the median exp(PRIOR_MU) ≈ 6 as a single uniform colour to make this
# explicit. The prior uncertainty (90% interval: ~0.1–340 ev/yr) is annotated
# in the title rather than shown spatially, because spatial variation would
# falsely imply the prior encodes geographic knowledge.
prior_lam_scalar = np.exp(PRIOR_MU)   # ≈ 6.05 ev/yr — median of LogNormal
prior_p05 = np.exp(PRIOR_MU - 1.645 * PRIOR_SIGMA_M1)
prior_p95 = np.exp(PRIOR_MU + 1.645 * PRIOR_SIGMA_M1)

# ---- Posterior mean per cell — Model 1 -------------------------------------
idata1_loaded = az.from_netcdf("../data/processed/idata_model1.nc")
post1_alpha = idata1_loaded.posterior["alpha"].mean(("chain","draw")).values
post1_lam   = np.exp(post1_alpha)

# ---- Posterior mean per cell — Model 2 -------------------------------------
post2_alpha = idata2.posterior["alpha"].mean(("chain","draw")).values
post2_lam   = np.exp(post2_alpha)

# ---- Build GeoDataFrame — ALL 208 cells so the prior map is fully tiled ----
cell_to_idx = {c: i for i, c in enumerate(cells)}  # active cells index
geoms, prior_vals, post1_vals, post2_vals, cell_ids = [], [], [], [], []
for lat_i in range(len(LAT_BINS) - 1):
    for lon_i in range(len(LON_BINS) - 1):
        cid = f"{lat_i}_{lon_i}"
        lat0, lat1 = LAT_BINS[lat_i], LAT_BINS[lat_i+1]
        lon0, lon1 = LON_BINS[lon_i], LON_BINS[lon_i+1]
        geoms.append(box(lon0, lat0, lon1, lat1))
        prior_vals.append(prior_lam_scalar)
        if cid in cell_to_idx:
            idx = cell_to_idx[cid]
            post1_vals.append(post1_lam[idx])
            post2_vals.append(post2_lam[idx])
        else:
            post1_vals.append(np.nan)             # inactive cell → grey
            post2_vals.append(np.nan)
        cell_ids.append(cid)

gdf = gpd.GeoDataFrame(
    {"cell_id": cell_ids, "prior_mean": prior_vals,
     "post1_mean": post1_vals, "post2_mean": post2_vals},
    geometry=geoms, crs="EPSG:4326")

# ---- Plot: prior | M1 posterior | M2 posterior -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 7), constrained_layout=True)

vmax = max(gdf["post1_mean"].max(), gdf["post2_mean"].max())
vmin = 0

# Prior panel: uniform colour = median of the shared prior (no geographic info)
import matplotlib.colors as mcolors
prior_grid = np.full((len(LAT_BINS)-1, len(LON_BINS)-1), prior_lam_scalar)
im0 = axes[0].imshow(prior_grid, origin="lower", aspect="auto",
                     interpolation="nearest",
                     extent=[LON_BINS[0], LON_BINS[-1], LAT_BINS[0], LAT_BINS[-1]],
                     cmap="YlOrRd", vmin=vmin, vmax=vmax)
axes[0].set_xlim(122, 154); axes[0].set_ylim(24, 50)
axes[0].set_title(
    f"Prior predictive median ≈ {prior_lam_scalar:.1f} ev/yr\n"
    f"(uniform — no geographic info; 90% CI: {prior_p05:.1f}–{prior_p95:.0f} ev/yr)",
    fontsize=10)
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
fig.colorbar(im0, ax=axes[0], label="Events / year", shrink=0.7)

gdf.plot(column="post1_mean", ax=axes[1], cmap="YlOrRd",
         vmin=vmin, vmax=vmax, legend=True,
         legend_kwds={"label": "Events / year", "shrink": 0.7})
axes[1].set_xlim(122, 154); axes[1].set_ylim(24, 50)
axes[1].set_title("Model 1 posterior mean\n(No Pooling)", fontsize=11)
axes[1].set_xlabel("Longitude")

gdf.plot(column="post2_mean", ax=axes[2], cmap="YlOrRd",
         vmin=vmin, vmax=vmax, legend=True,
         legend_kwds={"label": "Events / year", "shrink": 0.7})
axes[2].set_xlim(122, 154); axes[2].set_ylim(24, 50)
axes[2].set_title("Model 2 posterior mean\n(Partial Pooling — shrinkage)", fontsize=11)
axes[2].set_xlabel("Longitude")

fig.suptitle("Prior vs Posterior — what the data taught each model",
             fontsize=13, fontweight="bold")
os.makedirs("../report/figures", exist_ok=True)
plt.savefig("../report/figures/05_prior_vs_posterior_map.png", dpi=130, bbox_inches="tight")
plt.show()
print("Model 2 map is smoother: data-poor cells (west) are pulled toward the global mean.")
print("Eastern coast cells are similarly hot in both posteriors — data-rich, unaffected by shrinkage.")

### 5.5 Shrinkage effect — Model 1 vs Model 2

In [ ]:
idata1_loaded = az.from_netcdf("../data/processed/idata_model1.nc")

alpha1_mean = az.summary(idata1_loaded, var_names=["alpha"])["mean"].values
alpha1_sd   = az.summary(idata1_loaded, var_names=["alpha"])["sd"].values
alpha2_mean = az.summary(idata2,        var_names=["alpha"])["mean"].values
alpha2_sd   = az.summary(idata2,        var_names=["alpha"])["sd"].values
mu_g_post   = float(idata2.posterior["mu_global"].mean())

cell_totals_arr = meta.set_index("cell_id").loc[cells, "total_events"].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

sc = axes[0].scatter(alpha1_mean, alpha2_mean, c=np.log1p(cell_totals_arr),
                     cmap="viridis", s=20, alpha=0.8)
plt.colorbar(sc, ax=axes[0], label="log(1+total events)")
axes[0].axline((0,0), slope=1, color="grey", ls="--", label="No shrinkage")
axes[0].axhline(mu_g_post, color="red", ls=":", lw=1, label=f"mu_global={mu_g_post:.2f}")
axes[0].set_xlabel("Model 1 alpha (no pooling)")
axes[0].set_ylabel("Model 2 alpha (partial pooling)")
axes[0].set_title("Shrinkage in alpha (log-scale)")
axes[0].legend(fontsize=8)

axes[1].scatter(cell_totals_arr, alpha1_sd - alpha2_sd,
                c=np.log1p(cell_totals_arr), cmap="viridis", s=20, alpha=0.8)
axes[1].axhline(0, color="grey", ls="--")
axes[1].set_xlabel("Total events in cell")
axes[1].set_ylabel("SD reduction (M1 − M2)")
axes[1].set_title("Uncertainty reduction from pooling")
axes[1].set_xscale("log")

plt.suptitle("Shrinkage effect: partial pooling pulls data-poor cells toward mu_global",
             fontsize=11)
plt.savefig("../report/figures/06_shrinkage.png", dpi=130, bbox_inches="tight")
plt.show()

poor_mask = cell_totals_arr < 10
print(f"Data-poor cells (<10 events total): {poor_mask.sum()}")
print(f"  M1 mean alpha: {alpha1_mean[poor_mask].mean():.3f}")
print(f"  M2 mean alpha: {alpha2_mean[poor_mask].mean():.3f}  (toward mu_global={mu_g_post:.2f})")
print(f"  SD reduction:  {((alpha1_sd[poor_mask]-alpha2_sd[poor_mask])/alpha1_sd[poor_mask]).mean():.1%}")

### 5.6 Posterior intensity map (GeoPandas)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
vmax_single = gdf["post2_mean"].max()
gdf.plot(column="post2_mean", ax=ax, cmap="YlOrRd", legend=True,
         vmin=0, vmax=vmax_single,
         legend_kwds={"label": "Posterior mean events/year", "shrink": 0.7})
ax.set_xlim(122, 154); ax.set_ylim(24, 50)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Model 2 — Posterior mean seismic intensity per 2°×2° cell")
plt.show()

In [ ]:
pathlib.Path("../data/processed/idata_model2.nc").unlink(missing_ok=True)
idata2.to_netcdf("../data/processed/idata_model2.nc", overwrite_existing=True, engine="netcdf4")
print("Saved idata_model2.nc")

**Summary — Model 2.** Centered parameterization + adapt_delta=0.95 → 0
divergences, R-hat = 1.000, ESS > 4000. sigma_global ≈ 2.35 confirms cells
genuinely differ. Data-poor cells are pulled ~11% in sd toward the global mean;
data-rich cells are unchanged. The prior vs posterior map shows the eastern coast
cells are substantially hotter in the posterior than the uniform prior — exactly
where the data are most informative.

---
## 6. Model Comparison

**What we do:** compare the two models using WAIC and PSIS-LOO, inspect
Pareto-k diagnostics, and give a final qualitative assessment.

In [ ]:
idata1_c = az.from_netcdf("../data/processed/idata_model1.nc")
idata2_c = az.from_netcdf("../data/processed/idata_model2.nc")
annual_c  = pd.read_csv("../data/processed/grid_annual_counts.csv")

### WAIC

In [ ]:
waic_compare = az.compare({"Model1": idata1_c, "Model2": idata2_c}, ic="waic")
display_df(waic_compare, caption="WAIC comparison")
az.plot_compare(waic_compare, figsize=(7, 3))
plt.title("WAIC comparison"); plt.tight_layout(); plt.show()

**WAIC discussion.** WAIC estimates out-of-sample predictive accuracy by penalising
model complexity (p_waic = effective number of parameters). When some observations
have very high variance in their pointwise log-likelihood, p_waic becomes unreliable
— exactly our situation with the 2011 Tohoku cluster.

### PSIS-LOO

In [ ]:
loo_compare = az.compare({"Model1": idata1_c, "Model2": idata2_c}, ic="loo")
display_df(loo_compare, caption="PSIS-LOO comparison")
az.plot_compare(loo_compare, figsize=(7, 3))
plt.title("PSIS-LOO comparison"); plt.tight_layout(); plt.show()

In [ ]:
loo1 = az.loo(idata1_c, pointwise=True)
loo2 = az.loo(idata2_c, pointwise=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, loo, title in zip(axes, [loo1, loo2], ["Model 1", "Model 2"]):
    k = loo.pareto_k.values
    obs_idx = np.arange(len(k))
    ax.scatter(obs_idx, k, s=4, alpha=0.5,
               c=["red" if ki > 0.7 else "steelblue" for ki in k])
    ax.axhline(0.7, color="orange", ls="--", label="k=0.7 threshold")
    ax.axhline(1.0, color="red",    ls=":",  label="k=1.0")
    ax.set_xlabel("Observation index"); ax.set_ylabel("Pareto k")
    ax.set_title(f"{title} — Pareto-k diagnostics")
    ax.legend(fontsize=8)
    print(f"{title}: k>0.7: {(k>0.7).sum()}, max k={k.max():.2f}")
plt.tight_layout()
plt.savefig("../report/figures/07_pareto_k.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Live verdict — computed from actual IC objects; never goes stale
w1 = az.waic(idata1_c); w2 = az.waic(idata2_c)
l1 = az.loo(idata1_c);  l2 = az.loo(idata2_c)

waic_winner = "Model 1" if w1.elpd_waic > w2.elpd_waic else "Model 2"
loo_winner  = "Model 1" if l1.elpd_loo  > l2.elpd_loo  else "Model 2"
waic_diff   = abs(w1.elpd_waic - w2.elpd_waic)
loo_diff    = abs(l1.elpd_loo  - l2.elpd_loo)

print("=" * 55)
print(f"  WAIC  → {waic_winner} better  (|Δelpd| = {waic_diff:.1f})")
print(f"  LOO   → {loo_winner}  better  (|Δelpd| = {loo_diff:.1f})")
if waic_winner == loo_winner:
    print(f"  Criteria AGREE: {waic_winner} preferred.")
else:
    print("  Criteria DISAGREE — near-tie, rankings unreliable.")
print("=" * 55)

### Final assessment

**WAIC and PSIS-LOO** separate the models by only ~1.5–2 SE, **both issue
warnings**, and they **disagree on which ranks first** — the order flips between
the two criteria. This is a statistical near-tie driven by the 2011 Tohoku
cluster (Pareto-k up to ~8, ~55 observations with k > 0.7). Both criteria are
dominated by a handful of extreme, misspecified observations that a stationary
Poisson model cannot describe.

**We prefer Model 2** for qualitative reasons:
- it regularises data-poor cells (shrinkage), giving more stable estimates;
- the hierarchy is *data-supported* (sigma_global ≈ 2.35 ≫ 0);
- it costs only one additional parameter (sigma_global);
- it yields the honest, smooth intensity map the use case needs.

The real next step is a **non-stationary** model (e.g. with a 2011 indicator or
a time trend) — that would reduce Pareto-k below 0.7 and allow the information
criteria to arbitrate cleanly.

---
## 7. Summary & Key Figures

### Problem
We model the **annual count of M ≥ 4.0 earthquakes per 2°×2° grid cell** over
Japan (2000–2023). Use case: seismic-hazard assessment / insurance /
infrastructure planning, where a *map with calibrated uncertainty* matters more
than a point estimate. Data: USGS Earthquake Catalog API (~33 000 events, 154
active cells, 2086 cell-year observations).

### The two models

| | **Model 1 — no pooling** | **Model 2 — partial pooling** |
|---|---|---|
| per-cell prior | $\alpha_c\sim\mathcal N(1.8,\,2.07)$ | $\alpha_c\sim\mathcal N(\mu_g,\sigma_g)$ |
| scale $\sigma$ | **fixed** | **estimated** ($\sigma_g\sim$ HalfNormal(1)) |
| cells | independent | borrow strength (shrinkage) |

### Priors
Derived **only from external knowledge** (no data-leakage): published Japan rate
(~1200 M≥4/yr, JMA/ISC catalogues pre-2000) ÷ grid geometry (208 cells) ⇒
centre $\log(5.8)\approx1.8$; shape from Gutenberg–Richter law (b≈1).
Prior sensitivity analysis confirms the posterior is robust to prior changes
(Räty et al. 2023, WAMBS checklist).

### Posteriors
Both models converge cleanly (no divergences, $\hat R<1.01$; Model 2 uses
centered parameterization + `adapt_delta=0.95`). Model 2 shrinkage pulls
data-poor cells toward the global mean; data-rich cells are unchanged.
`sigma_global` ≈ 2.35 confirms the hierarchy is data-supported.

### Model comparison
WAIC and PSIS-LOO separate the models by only ~1.5–2 SE, both warn, and
disagree on ranking — dominated by 2011 Tohoku (Pareto-k up to ~8).
We prefer **Model 2**: regularises data-poor cells, data-supported hierarchy,
one extra parameter, honest uncertainty map.

In [ ]:
from utils.display import display_image

figures = [
    ("../report/figures/01_raw_map.png",              520, "Fig 1 — Raw earthquake epicentres (input data)"),
    ("../report/figures/02_annual_counts.png",         520, "Fig 2 — Annual event count (2011 Tohoku outlier)"),
    ("../report/figures/03_dag.png",                   620, "Fig 3 — Generative DAGs (Model 1 & 2)"),
    ("../report/figures/04_prior_predictive_check.png",520, "Fig 4 — Prior predictive check 2"),
    ("../report/figures/05_prior_vs_posterior_map.png",700, "Fig 5 — Prior vs Posterior intensity map"),
    ("../report/figures/06_shrinkage.png",             620, "Fig 6 — Shrinkage effect (M1 vs M2)"),
    ("../report/figures/07_pareto_k.png",              620, "Fig 7 — Pareto-k diagnostics (PSIS-LOO)"),
]

for path, width, caption in figures:
    print(f"\n{'='*60}")
    print(caption)
    print('='*60)
    display_image(path, width=width)